In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

ratings_path = "ratings.csv"
df = pd.read_csv(ratings_path)
df = df[["userId","movieId","rating"]].dropna()

user_ids = df["userId"].unique()
movie_ids = df["movieId"].unique()
user2idx = {u:i for i,u in enumerate(sorted(user_ids))}
movie2idx = {m:i for i,m in enumerate(sorted(movie_ids))}
df["user_idx"] = df["userId"].map(user2idx)
df["movie_idx"] = df["movieId"].map(movie2idx)

def split_by_user(data, val_ratio=0.1):
    groups = data.groupby("user_idx")
    train_parts = []
    val_parts = []
    for uid, g in groups:
        g = g.sample(frac=1.0, random_state=seed)
        n_val = max(1, int(len(g)*val_ratio))
        val_parts.append(g.iloc[:n_val])
        train_parts.append(g.iloc[n_val:])
    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    return train_df, val_df

train_df, val_df = split_by_user(df, val_ratio=0.1)

class RatingsDS(Dataset):
    def __init__(self, data):
        self.u = torch.tensor(data["user_idx"].values, dtype=torch.long)
        self.m = torch.tensor(data["movie_idx"].values, dtype=torch.long)
        self.r = torch.tensor(data["rating"].values, dtype=torch.float32)
    def __len__(self):
        return len(self.r)
    def __getitem__(self, i):
        return self.u[i], self.m[i], self.r[i]

train_ds = RatingsDS(train_df)
val_ds = RatingsDS(val_df)

batch_size = 1024
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

n_users = len(user2idx)
n_movies = len(movie2idx)

class Model(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, p=0.3):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        self.dropout = nn.Dropout(p)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*2, 256),
            nn.LeakyReLU(),
            nn.Dropout(p),
            nn.Linear(256, 128),
            nn.LeakyReLU(),
            nn.Dropout(p),
            nn.Linear(128, 64),
            nn.LeakyReLU(),
            nn.Dropout(p),
            nn.Linear(64, 1)
        )
    def forward(self, u, i):
        ue = self.user_emb(u)
        ie = self.item_emb(i)
        x = torch.cat([ue, ie], dim=1)
        x = self.dropout(x)
        out = self.mlp(x)
        return out.squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(n_users, n_movies, emb_dim=16, p=0.3).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def rmse(y_true, y_pred):
    return math.sqrt(((y_true - y_pred)**2).mean().item())

best_val = float("inf")
epochs = 30
train_mse_hist = []
val_rmse_hist = []

for epoch in range(1, epochs+1):
    model.train()
    train_losses = []
    for u,m,r in train_loader:
        u = u.to(device); m = m.to(device); r = r.to(device)
        pred = model(u,m)
        loss = criterion(pred, r)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    train_mse = float(np.mean(train_losses))
    model.eval()
    with torch.no_grad():
        ys = []
        ps = []
        for u,m,r in val_loader:
            u = u.to(device); m = m.to(device)
            p = model(u,m)
            ys.append(r)
            ps.append(p.cpu())
        y = torch.cat(ys).cpu()
        p = torch.cat(ps).cpu()
        val_rmse = rmse(y, p)
    train_mse_hist.append(train_mse)
    val_rmse_hist.append(val_rmse)
    print(f"Epoch {epoch} | train_mse={train_mse:.4f} | val_RMSE={val_rmse:.4f}")
    if val_rmse < best_val:
        best_val = val_rmse

print("Mejor val_RMSE:", best_val)

plt.figure(figsize=(7,4))
plt.plot(range(1, epochs+1), train_mse_hist, marker="o", label="Train MSE")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Evolución del MSE de entrenamiento")
plt.grid(True, linestyle=":")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(range(1, epochs+1), val_rmse_hist, marker="o", label="Val RMSE")
plt.xlabel("Época")
plt.ylabel("RMSE")
plt.title("Evolución del RMSE de validación")
plt.grid(True, linestyle=":")
plt.legend()
plt.tight_layout()
plt.show()

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for u, m, r in val_loader:
        u, m = u.to(device), m.to(device)
        preds = model(u, m).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(r.numpy())

r2 = r2_score(all_true, all_preds)
print(f"R2 validación: {r2:.4f}")

plt.figure(figsize=(5,5))
plt.scatter(all_true, all_preds, alpha=0.4)
plt.plot([0,5],[0,5], linestyle="--")
plt.xlabel("Rating real")
plt.ylabel("Rating predicho")
plt.title(f"Predicho vs Real (Validación) | R2={r2:.4f}")
plt.grid(True, linestyle=":")
plt.tight_layout()
plt.show()
